In [16]:
#run refresh_trade_views to init arbitrage tables
#from db.auto_repo_sqlite import snapshot_many, TableSpec, upsert_many
#from domain.market_rows import MarketGoodRow, MarketTransactionRow
#import sqlite3


from __future__ import annotations
import asyncio
from typing import Optional, List
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi

from runtime_support import (
    setup_client_from_env,
    api_navigate_ship,
    api_get_ship_nav,
    api_patch_nav_flight_mode,
    api_purchase_cargo,
    api_sell_cargo,
    build_fleet_object
    )

from core_helpers import init_world_state, load_initial_fleet_state

from core_helpers import build_nodes_from_traits_dict
from refuel_routing import plan_route_and_refuel_with_reserve

from db.refresh_trade_views import refresh_trade_views
from services.arbitrage_repo import top_arbitrage

async def trade_loop(
    fleet_api,
    ship,
    route_buy_point,
    route_sell_point,
    arbi_buy_wp,
    arbi_sell_wp,
    arbi_trade_symbol,
    units=20
):
    #while True:  # repeat forever, or replace with a counter/condition

        # Navigate to buy waypoint(s)
        for rt in route_buy_point:
            await api_navigate_ship(fleet_api, ship, rt)

        # Dock and buy
        await api_purchase_cargo(fleet_api, ship, arbi_buy_wp, arbi_trade_symbol, units)
        await api_purchase_cargo(fleet_api, ship, arbi_buy_wp, arbi_trade_symbol, units)

        # Navigate to sell waypoint(s)
        for rt in route_sell_point:
            await api_navigate_ship(fleet_api, ship, rt)

        # Dock and sell
        await api_sell_cargo(fleet_api, ship, arbi_sell_wp, arbi_trade_symbol, units)
        await api_sell_cargo(fleet_api, ship, arbi_sell_wp, arbi_trade_symbol, units)
        trade_refresh

def check_cost(route_list: list):
    if not route_list:
        return(0)
    else:
        total = sum(int(rl[-1]) for rl in route_list)
        return(int(total))

def a_b_dist(coord1: str, coord2: str):

    import math
    x1 = world_state.waypoints.by_symbol[coord1].x
    y1 = world_state.waypoints.by_symbol[coord1].y
    x2 = world_state.waypoints.by_symbol[coord2].x
    y2 = world_state.waypoints.by_symbol[coord2].y

    x = x1-x2
    y = y1-y2

    return(math.hypot(x,y))

#---------------------------------------------------------

with setup_client_from_env() as client:
    fleet_api = FleetApi(client)
    agents_api = AgentsApi(client)
    systems_api = SystemsApi(client)


world_state = await init_world_state(fleet_api, agents_api, systems_api)

# --------- define ship roles -----------
fleet_state = await load_initial_fleet_state(fleet_api)

# Extract symbol for a given role
def get_symbol_by_role(ships_dict, role):
    for ship in ships_dict.values():
        if ship.role == role:
            return ship.symbol
    return None

command_ship = get_symbol_by_role(fleet_state.specs, "COMMAND")
satellite = get_symbol_by_role(fleet_state.specs, "SATELLITE")


In [3]:

# init arbitrage and trade views tables
trade_refresh = refresh_trade_views(lookback_hours=48, create_schema=True)

# get ranked arbitrage data
arbitrage_rank = top_arbitrage()
print(arbitrage_rank)

        trade_symbol buy_waypoint  buy_price sell_waypoint  sell_price  delta  \
0     ASSAULT_RIFLES   X1-XG6-E50       2617    X1-XG6-J63        3848   1231   
1            JEWELRY   X1-XG6-H59       2316     X1-XG6-A1        3503   1187   
2    MICROPROCESSORS    X1-XG6-A3       2420    X1-XG6-D49        3567   1147   
3           CLOTHING   X1-XG6-K89       3721    X1-XG6-J63        4623    902   
4           FIREARMS   X1-XG6-E50       2561    X1-XG6-J63        3456    895   
5           MEDICINE   X1-XG6-D48       3733     X1-XG6-A1        4210    477   
6               GOLD    X1-XG6-B7        214    X1-XG6-H59         310     96   
7             SILVER    X1-XG6-B7        213    X1-XG6-H59         290     77   
8               FUEL   X1-XG6-G55         53    X1-XG6-J63          68     15   
9   SILICON_CRYSTALS    X1-XG6-B7         37     X1-XG6-A3          42      5   
10   LIQUID_HYDROGEN   X1-XG6-C46         27    X1-XG6-F52          31      4   
11   LIQUID_NITROGEN   X1-XG

In [4]:
#select first row in arbitrage table

top_arbi = arbitrage_rank.iloc[0].values.tolist()

# extract values to pass to profitability calcs

arb_trade_symbol = top_arbi[0]
arb_buy_wp = top_arbi[1]
buy_price = top_arbi[2]
arb_sell_wp = top_arbi[3]
sell_price = top_arbi[4]

cargo_capacity = fleet_state.activities[command_ship].cargo_capacity

# profit from arbitrage is calculated
arbitrage_profit = (sell_price-buy_price)*cargo_capacity



In [17]:
world_state = await init_world_state(fleet_api, agents_api, systems_api)

wps = world_state.traits.by_wp
nodes = build_nodes_from_traits_dict(wps)

fleet_state = await load_initial_fleet_state(fleet_api)
cur_wp = fleet_state.activities[command_ship].current_waypoint
fuel_capacity = fleet_state.activities[command_ship].fuel_capacity
cargo_fuel_capacity = fleet_state.activities[command_ship].cargo_capacity - fleet_state.activities[command_ship].cargo_units
current_tank = fleet_state.activities[command_ship].fuel_current

plan_to_start = plan_route_and_refuel_with_reserve(nodes, cur_wp, arb_buy_wp, fuel_capacity, cargo_fuel_capacity, 1, 3600, current_tank, 0, 500, 0, 5, 0, 1 )
plan_to_sell = plan_route_and_refuel_with_reserve(nodes, arb_buy_wp, arb_sell_wp, fuel_capacity, cargo_fuel_capacity, 1, 3600, current_tank, 0, 100, 0, 5, 0, 1 )

start_cost = plan_to_start.get('actions')
sell_cost = plan_to_sell.get('actions')

travel_cost = check_cost(start_cost) + check_cost(sell_cost)

route_buy_point = plan_to_start["path"]
route_sell_point = plan_to_sell["path"]

travel_cost

1070

In [19]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Node:
    symbol: str
    x: float
    y: float
    has_fuel: bool = False
    price: Optional[float] = None  # price per fuel unit if known

from typing import Iterable, List, Tuple

async def travel_cost(nodes: Iterable[Node], ship_symbol: str, top_arbi: list,
)-> Tuple[float, List[str], List[str], str, str, str]:

    fleet_state = await load_initial_fleet_state(fleet_api)
    ss = fleet_state.activities[ship_symbol]

    cur_wp = ss.current_waypoint
    fuel_capacity = ss.fuel_capacity
    cargo_fuel_capacity = ss.cargo_capacity - fleet_state.activities[ship_symbol].cargo_units
    current_tank = ss.fuel_current

    arb_buy_wp = top_arbi[1]
    arb_sell_wp = top_arbi[3]

    plan_to_start = plan_route_and_refuel_with_reserve(nodes, cur_wp, arb_buy_wp, fuel_capacity, cargo_fuel_capacity, 1, 3600, current_tank, 0, 100, 0, 5, 0, 1 )
    plan_to_sell = plan_route_and_refuel_with_reserve(nodes, arb_buy_wp, arb_sell_wp, fuel_capacity, cargo_fuel_capacity, 1, 3600, current_tank, 0, 100, 0, 5, 0, 1 )

    start_cost = plan_to_start.get('actions') # the cost of arriving at the buying waypoint
    sell_cost = plan_to_sell.get('actions') # the cost of travelling from the buy point to the sell point

    travel_cost = check_cost(start_cost) + check_cost(sell_cost)

    route_buy_point = plan_to_start["path"]
    route_sell_point = plan_to_sell["path"]
    
    return(travel_cost, route_buy_point, route_sell_point, arb_buy_wp, arb_sell_wp)


async def arbitrage_profit(ship_symbol: str, top_arbi: list):    

    # extract values to pass to profitability calcs
    #select first row in arbitrage table
    buy_price = top_arbi[2]
    sell_price = top_arbi[4]

    cargo_capacity = fleet_state.activities[command_ship].cargo_capacity

    # profit from arbitrage is calculated
    arbitrage_profit = (sell_price-buy_price)*cargo_capacity

    return(int(arbitrage_profit))

In [20]:
async def trade_route_decision(ship_symbol: str, nodes: Node):

    # init arbitrage and trade views tables
    refresh_trade_views(lookback_hours=48, create_schema=True)

    # get ranked arbitrage data
    arbitrage_rank = top_arbitrage()

    top_arbi = arbitrage_rank.iloc[0].values.tolist()
    
    arb_profit = await arbitrage_profit(ship_symbol, top_arbi)
    trav_cost, route_buy_point, route_sell_point, arb_buy_wp, arb_sell_wp = await travel_cost(nodes, ship_symbol, top_arbi)

    balance = arb_profit - trav_cost

    if balance > 0:
        print("Profit on this route is: ", balance, " with a travel cost of: ", trav_cost)

        arb_trade_symbol = top_arbi[0]
        await trade_loop(fleet_api,
            ship_symbol,
            route_buy_point,
            route_sell_point,
            arb_buy_wp,
            arb_sell_wp,
            arb_trade_symbol,
            units=20)
    else:
        print("This trade route would result in a loss of: ", balance, " trying next descending position arbitrage trade symbol")

In [ ]:
while True:
    await trade_route_decision(command_ship, nodes)

Profit on this route is:  48170  with a travel cost of:  1070
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Ship is already at the destination
Prep complete
[SKIP] Navigation aborted, already at destination
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Refuelling now...
Not in orbit, going into orbit now...
Prep complete
DDDD-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 290.923987


In [82]:
from typing import Iterable, List, Tuple, Dict, Any

# Assumptions:
# - refresh_trade_views, top_arbitrage, arbitrage_profit are sync functions.
# - travel_cost is async and returns: Tuple[float, List[str], List[str], str, str]
#     -> (travel_cost, route_buy_point, route_sell_point, arb_buy_wp, arb_sell_wp)
# - trade_loop is async and executes one full buy/sell cycle for a given trade symbol.

async def trade_route_decision(
    fleet_api,
    ship_symbol: str,
    nodes: Iterable[Node],
    *,
    lookback_hours: int = 48,
    units: int = 20,
) -> Dict[str, Any]:
    """
    Try arbitrage rows in ranked order. For each row:
      - compute arbitrage profit for the ship
      - compute travel cost/route
      - if (profit - travel_cost) > 0: run the trade loop
    Continue iterating after each trade.
    If none are profitable, print a message and exit.

    Returns a summary dict with per-row results.
    """

    # 1) Ensure views are fresh (consider moving to a separate scheduler if this can be slow)
    refresh_trade_views(lookback_hours=lookback_hours, create_schema=True)

    # 2) Get ranked arbitrage candidates (pandas DataFrame)
    arbitrage_rank = top_arbitrage()
    if arbitrage_rank is None or getattr(arbitrage_rank, "empty", False):
        print("no profitable arbitrage currently available (no rows)")
        return {"executed": [], "skipped": [], "message": "empty arbitrage_rank"}

    executed: List[Dict[str, Any]] = []
    skipped: List[Dict[str, Any]] = []

    # 3) Walk the table top-to-bottom
    for idx, row in arbitrage_rank.iterrows():
        # Convert row to a structure your helpers expect
        top_arbi = row.values.tolist()

        # Extract fields you’ll need (adjust indices to match your schema)
        # Commonly: [trade_symbol, buy_wp, sell_wp, buy_price, sell_price, ...]
        try:
            arb_trade_symbol = top_arbi[0]
        except Exception:
            arb_trade_symbol = row.get("trade_symbol") if hasattr(row, "get") else str(top_arbi[0])

        # Compute profit for this ship & opportunity
        try:
            arbi_profit_val = arbitrage_profit(ship_symbol, top_arbi)
        except Exception as e:
            skipped.append({"idx": idx, "trade_symbol": arb_trade_symbol, "reason": f"profit error: {e}"})
            continue

        # Compute travel plan/cost (async)
        try:
            travel_cost_val, route_buy_point, route_sell_point, arb_buy_wp, arb_sell_wp = await travel_cost(
                nodes, ship_symbol, top_arbi
            )
        except Exception as e:
            skipped.append({"idx": idx, "trade_symbol": arb_trade_symbol, "reason": f"travel_cost error: {e}"})
            continue

        balance = arbi_profit_val - travel_cost_val

        if balance > 0:
            print(f"[{idx}] PROFIT {arb_trade_symbol}: balance={balance:.2f} (profit={arbi_profit_val:.2f}, travel={travel_cost_val:.2f})")
            try:
                await trade_loop(
                    fleet_api,
                    ship_symbol,
                    route_buy_point,
                    route_sell_point,
                    arb_buy_wp,
                    arb_sell_wp,
                    arb_trade_symbol,
                    units=units,
                )
                executed.append({
                    "idx": idx,
                    "trade_symbol": arb_trade_symbol,
                    "balance": balance,
                    "profit": arbi_profit_val,
                    "travel_cost": travel_cost_val,
                    "buy_route": route_buy_point,
                    "sell_route": route_sell_point,
                })
            except Exception as e:
                skipped.append({"idx": idx, "trade_symbol": arb_trade_symbol, "reason": f"trade_loop error: {e}"})
                # keep scanning other rows
        else:
            print(f"[{idx}] SKIP {arb_trade_symbol}: not profitable (balance={balance:.2f})")
            skipped.append({
                "idx": idx,
                "trade_symbol": arb_trade_symbol,
                "balance": balance,
                "profit": arbi_profit_val,
                "travel_cost": travel_cost_val,
                "reason": "non-positive balance",
            })

    # Final outcome
    if not executed:
        print("no profitable arbitrage currently available")
        message = "none executed"
    else:
        message = f"executed {len(executed)} trade(s)"

    return {"executed": executed, "skipped": skipped, "message": message}


In [80]:
if arbitrage_profit - travel_cost > 0:
    print("Profit on this route is: ", arbitrage_profit - travel_cost, " with a travel cost of: ", travel_cost)
    await trade_loop(fleet_api,
        command_ship,
        route_buy_point,
        route_sell_point,
        arb_buy_wp,
        arb_sell_wp,
        arb_trade_symbol,
        units=20)
else:
    print("This trade route woud result in a loss of: ", arbitrage_profit - travel_cost)

Profit on this route is:  52918  with a travel cost of:  2
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Ship is already at the destination
Prep complete
[SKIP] Navigation aborted, already at destination
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Refuelling now...
Not in orbit, going into orbit now...
Prep complete
DDDD-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 200.879896
Arrived and ready
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Refuelling now...
Not in orbit, going into orbit now...
Prep complete
DDDD-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 290.913153
Arrived and ready
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Ship is already at the destination
Prep complete
[SKIP] Navigation aborted, already at destination
starting navigation

In [72]:
"""import matplotlib.pyplot as plt
import networkx as nx

G = plan_to_start["graph"]

# Use the x,y coords stored in Node
pos = {n: (G.nodes[n]["data"].x, G.nodes[n]["data"].y) for n in G.nodes}

plt.figure(figsize=(8, 6))
nx.draw(
    G, pos,
    with_labels=True,
    node_size=500,
    node_color="lightblue",
    font_size=8,
    arrows=True
)

# Optionally label edge distances
edge_labels = {(u, v): f'{d["distance"]:.1f}' for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)

plt.show()
"""

'import matplotlib.pyplot as plt\nimport networkx as nx\n\nG = plan_to_start["graph"]\n\n# Use the x,y coords stored in Node\npos = {n: (G.nodes[n]["data"].x, G.nodes[n]["data"].y) for n in G.nodes}\n\nplt.figure(figsize=(8, 6))\nnx.draw(\n    G, pos,\n    with_labels=True,\n    node_size=500,\n    node_color="lightblue",\n    font_size=8,\n    arrows=True\n)\n\n# Optionally label edge distances\nedge_labels = {(u, v): f\'{d["distance"]:.1f}\' for u, v, d in G.edges(data=True)}\nnx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)\n\nplt.show()\n'